In [1]:
# Step 4 Experiment 3 — Corrected Random Representative-Date Benchmark
# COMPLETE JUPYTER-LAB PYTHON CODE IN ONE BLOCK
#
# Expected input files in current working directory:
#   experiment_input.zip
#   tda_step4_exp1_outputs.zip
#   tda_step4_exp2_outputs.zip
#
# Output:
#   tda_step4_exp3_outputs/
#   tda_step4_exp3_outputs.zip

import os
import io
import re
import json
import math
import shutil
import zipfile
import hashlib
import warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
from pandas.errors import EmptyDataError

warnings.filterwarnings("ignore")

# ============================================================
# 0. CONFIG
# ============================================================

EXPERIMENT_INPUT_ZIP = Path("experiment_input.zip")
EXP1_ZIP = Path("tda_step4_exp1_outputs.zip")
EXP2_ZIP = Path("tda_step4_exp2_outputs.zip")

OUTPUT_DIR = Path("tda_step4_exp3_outputs")
METADATA_DIR = OUTPUT_DIR / "00_metadata"
WORK_DIR = Path("_step4_exp3_work")

N_RANDOM_TRIALS = 100
SEED_BASE = 42

H = 21
L = 0.20
K = 3
WINDOW_LOOKBACK = 504
TRANSACTION_COST_RATE = 0.001
DEFAULT_INITIAL_NAV = 1_000_000.0
TRADING_DAYS_PER_YEAR = 252
CONSISTENCY_TOL = 1e-8

np.random.seed(SEED_BASE)

# ============================================================
# 1. GENERAL UTILITIES
# ============================================================

def sha256_file(path: Path) -> str:
    path = Path(path)
    if not path.exists() or not path.is_file():
        return ""
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def clean_dir(path: Path):
    path = Path(path)
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)

def extract_zip_recursive(zip_path: Path, dest_dir: Path):
    zip_path = Path(zip_path)
    dest_dir = Path(dest_dir)

    if not zip_path.exists():
        raise FileNotFoundError(f"Required zip not found: {zip_path}")

    dest_dir.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(dest_dir)

    processed = set()
    while True:
        nested_zips = [p for p in dest_dir.rglob("*.zip") if p not in processed]
        if not nested_zips:
            break

        for nested_zip in nested_zips:
            processed.add(nested_zip)
            nested_dest = nested_zip.parent / f"{nested_zip.stem}_extracted"
            if not nested_dest.exists():
                nested_dest.mkdir(parents=True, exist_ok=True)
                try:
                    with zipfile.ZipFile(nested_zip, "r") as z:
                        z.extractall(nested_dest)
                except zipfile.BadZipFile:
                    print(f"WARNING: Bad nested zip skipped: {nested_zip}")

def find_file(root: Path, filename: str):
    root = Path(root)
    matches = list(root.rglob(filename))
    if not matches:
        return None
    return sorted(matches, key=lambda p: (len(str(p)), str(p)))[0]

def find_files(root: Path, filename: str):
    root = Path(root)
    return sorted(root.rglob(filename), key=lambda p: (len(str(p)), str(p)))

def safe_read_csv(path: Path, default_columns=None):
    """
    Robust CSV reader:
    - Handles missing files
    - Handles zero-byte files
    - Handles CSV files with no columns
    """
    path = Path(path)

    if default_columns is None:
        default_columns = []

    if path is None or not path.exists():
        return pd.DataFrame(columns=default_columns)

    try:
        if path.stat().st_size == 0:
            return pd.DataFrame(columns=default_columns)

        df = pd.read_csv(path)

        if df.empty and len(df.columns) == 0:
            return pd.DataFrame(columns=default_columns)

        return df

    except EmptyDataError:
        return pd.DataFrame(columns=default_columns)
    except Exception as e:
        return pd.DataFrame([{
            "read_error": str(e),
            "source_path": str(path)
        }])

def read_csv_with_date(path: Path):
    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(f"CSV file not found: {path}")

    if path.stat().st_size == 0:
        raise EmptyDataError(f"CSV file is empty: {path}")

    df = pd.read_csv(path)

    if df.empty and len(df.columns) == 0:
        raise EmptyDataError(f"CSV file has no columns: {path}")

    possible_date_cols = ["Date", "date", "Datetime", "datetime", "Unnamed: 0"]
    date_col = None

    for c in possible_date_cols:
        if c in df.columns:
            date_col = c
            break

    if date_col is None:
        date_col = df.columns[0]

    try:
        parsed_dates = pd.to_datetime(df[date_col], errors="coerce")
        if parsed_dates.notna().mean() > 0.90:
            df = df.drop(columns=[date_col])
            df.index = parsed_dates
            df.index.name = "Date"
    except Exception:
        pass

    df = df.apply(pd.to_numeric, errors="coerce")
    df = df.sort_index()

    if isinstance(df.index, pd.DatetimeIndex):
        df = df[~df.index.duplicated(keep="first")]

    return df

def save_df_with_date_index(df: pd.DataFrame, path: Path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    out = df.copy()
    out.index.name = "Date"
    out.to_csv(path)

def max_drawdown_from_nav(nav_series: pd.Series) -> float:
    nav = pd.Series(nav_series).dropna().astype(float)
    if len(nav) == 0:
        return np.nan
    running_max = nav.cummax()
    dd = nav / running_max - 1.0
    return float(dd.min())

def annualized_return_from_daily_returns(daily_returns: pd.Series) -> float:
    r = pd.Series(daily_returns).replace([np.inf, -np.inf], np.nan).dropna().astype(float)
    if len(r) == 0:
        return np.nan

    cumulative = float((1.0 + r).prod() - 1.0)
    years = len(r) / TRADING_DAYS_PER_YEAR

    if years <= 0:
        return np.nan

    if cumulative <= -1.0:
        return -1.0

    return float((1.0 + cumulative) ** (1.0 / years) - 1.0)

def annualized_sharpe_from_daily_returns(daily_returns: pd.Series) -> float:
    r = pd.Series(daily_returns).replace([np.inf, -np.inf], np.nan).dropna().astype(float)
    if len(r) < 2:
        return np.nan

    std = r.std(ddof=1)
    if std == 0 or np.isnan(std):
        return np.nan

    return float(np.sqrt(TRADING_DAYS_PER_YEAR) * r.mean() / std)

def one_way_turnover(new_weights: dict, old_weights: dict) -> float:
    all_names = set(new_weights.keys()) | set(old_weights.keys())
    return float(0.5 * sum(abs(new_weights.get(x, 0.0) - old_weights.get(x, 0.0)) for x in all_names))

def equal_weights(stocks):
    stocks = sorted(list(stocks))
    if len(stocks) == 0:
        return {}
    w = 1.0 / len(stocks)
    return {s: w for s in stocks}

def percentile_vs_distribution(value, distribution, higher_is_better=True):
    dist = pd.Series(distribution).replace([np.inf, -np.inf], np.nan).dropna().astype(float)

    if len(dist) == 0 or pd.isna(value):
        return np.nan

    if higher_is_better:
        return float((dist <= value).mean() * 100.0)
    else:
        return float((dist >= value).mean() * 100.0)

def make_artifact_check(output_dir: Path, required_rel_paths):
    rows = []
    output_dir = Path(output_dir)

    for rel in required_rel_paths:
        p = output_dir / rel
        rows.append({
            "file": rel,
            "exists": bool(p.exists()),
            "size_bytes": int(p.stat().st_size) if p.exists() else 0
        })

    return pd.DataFrame(rows)

def normalize_ticker_columns(df: pd.DataFrame, valid_tickers):
    valid_tickers = list(valid_tickers)

    keep = [c for c in df.columns if c in valid_tickers]
    df = df.loc[:, keep].copy()

    df = df.loc[:, ~df.columns.duplicated()]
    df = df.sort_index()

    return df

def safe_to_markdown(df: pd.DataFrame, index=False):
    try:
        return df.to_markdown(index=index)
    except Exception:
        return df.to_string(index=index)

def get_col_case_insensitive(df: pd.DataFrame, candidates):
    lower_map = {str(c).lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in lower_map:
            return lower_map[cand.lower()]
    return None

# ============================================================
# 2. VALIDATE INPUTS AND EXTRACT
# ============================================================

if not EXPERIMENT_INPUT_ZIP.exists():
    raise FileNotFoundError("Required experiment_input.zip not found in current working directory.")

if not EXP1_ZIP.exists():
    raise FileNotFoundError("Required tda_step4_exp1_outputs.zip not found in current working directory.")

if not EXP2_ZIP.exists():
    raise FileNotFoundError("Required tda_step4_exp2_outputs.zip not found in current working directory.")

clean_dir(WORK_DIR)
clean_dir(OUTPUT_DIR)
METADATA_DIR.mkdir(parents=True, exist_ok=True)

EXP_INPUT_DIR = WORK_DIR / "experiment_input_extracted"
EXP1_DIR = WORK_DIR / "exp1_extracted"
EXP2_DIR = WORK_DIR / "exp2_extracted"

extract_zip_recursive(EXPERIMENT_INPUT_ZIP, EXP_INPUT_DIR)
extract_zip_recursive(EXP1_ZIP, EXP1_DIR)
extract_zip_recursive(EXP2_ZIP, EXP2_DIR)

print("Input extraction completed.")
print(f"- experiment_input extracted to: {EXP_INPUT_DIR}")
print(f"- exp1 outputs extracted to: {EXP1_DIR}")
print(f"- exp2 outputs extracted to: {EXP2_DIR}")

# ============================================================
# 3. LOAD STEP 1 METADATA
# ============================================================

step1_meta_path = find_file(EXP_INPUT_DIR, "step1_experiment_metadata.json")

if step1_meta_path is None:
    raise FileNotFoundError(
        "step1_experiment_metadata.json not found inside experiment_input.zip or nested zips. "
        "This is required for valid_tickers, dates, and core experiment metadata."
    )

with open(step1_meta_path, "r", encoding="utf-8") as f:
    step1_meta = json.load(f)

valid_tickers = step1_meta.get("valid_tickers", None)

if not valid_tickers:
    raise ValueError("valid_tickers missing or empty in step1_experiment_metadata.json.")

start_date = step1_meta.get("start_date", None)
end_date = step1_meta.get("end_date", None)

metadata_window_lookback = int(step1_meta.get("window_lookback", WINDOW_LOOKBACK))
metadata_k = int(step1_meta.get("k", K))

if metadata_window_lookback != WINDOW_LOOKBACK:
    print(
        f"WARNING: Step 1 metadata window_lookback={metadata_window_lookback}; "
        f"Exp3 core requirement uses {WINDOW_LOOKBACK}."
    )

if metadata_k != K:
    print(
        f"WARNING: Step 1 metadata k={metadata_k}; "
        f"Exp3 core requirement uses {K}."
    )

initial_nav = float(step1_meta.get("initial_nav", DEFAULT_INITIAL_NAV))

print("\nLoaded Step 1 metadata:")
print(f"- step1 metadata path: {step1_meta_path}")
print(f"- valid ticker count: {len(valid_tickers)}")
print(f"- start_date: {start_date}")
print(f"- end_date: {end_date}")
print(f"- initial_nav: {initial_nav}")

# ============================================================
# 4. LOAD OR DOWNLOAD STOCK-LEVEL RETURNS
# ============================================================

returns_source_path = find_file(EXP2_DIR, "downloaded_stock_returns_yfinance_auto_adjust_true.csv")
prices_source_path = find_file(EXP2_DIR, "downloaded_stock_prices_yfinance_auto_adjust_true.csv")
failed_batches_source_path = find_file(EXP2_DIR, "failed_yfinance_batches.csv")

if returns_source_path is None:
    returns_source_path = find_file(EXP_INPUT_DIR, "downloaded_stock_returns_yfinance_auto_adjust_true.csv")

if prices_source_path is None:
    prices_source_path = find_file(EXP_INPUT_DIR, "downloaded_stock_prices_yfinance_auto_adjust_true.csv")

if failed_batches_source_path is None:
    failed_batches_source_path = find_file(EXP_INPUT_DIR, "failed_yfinance_batches.csv")

data_source_note = ""

if returns_source_path is not None:
    stock_returns = read_csv_with_date(returns_source_path)
    stock_returns = normalize_ticker_columns(stock_returns, valid_tickers)
    data_source_note = f"Loaded existing returns matrix from {returns_source_path}"
    print(f"\nLoaded stock returns from: {returns_source_path}")

else:
    print("\nNo existing stock returns matrix found. Attempting yfinance download...")

    try:
        import yfinance as yf
    except ImportError as e:
        raise ImportError(
            "yfinance is required to download missing stock-level prices/returns. "
            "Please install yfinance or provide downloaded_stock_returns_yfinance_auto_adjust_true.csv."
        ) from e

    if start_date is None or end_date is None:
        raise ValueError(
            "start_date/end_date missing from Step 1 metadata; cannot download yfinance data."
        )

    yf_tickers = [t.replace(".", "-") for t in valid_tickers]
    reverse_map = {t.replace(".", "-"): t for t in valid_tickers}

    all_prices = []
    failed_batches = []
    batch_size = 50

    for i in range(0, len(yf_tickers), batch_size):
        batch = yf_tickers[i:i + batch_size]

        try:
            data = yf.download(
                tickers=batch,
                start=start_date,
                end=end_date,
                auto_adjust=True,
                progress=False,
                group_by="column",
                threads=True
            )

            if data is None or data.empty:
                raise ValueError("Empty yfinance response.")

            if isinstance(data.columns, pd.MultiIndex):
                if "Close" in data.columns.get_level_values(0):
                    px = data["Close"].copy()
                elif "Adj Close" in data.columns.get_level_values(0):
                    px = data["Adj Close"].copy()
                else:
                    raise ValueError("No Close or Adj Close in yfinance response.")
            else:
                if "Close" not in data.columns:
                    raise ValueError("No Close column in yfinance response.")
                px = data[["Close"]].copy()
                px.columns = batch

            px = px.rename(columns=reverse_map)
            all_prices.append(px)

        except Exception as e:
            failed_batches.append({
                "batch_start": i,
                "tickers": ",".join(batch),
                "error": str(e)
            })

    if not all_prices:
        raise RuntimeError("No prices downloaded from yfinance.")

    stock_prices = pd.concat(all_prices, axis=1)
    stock_prices = stock_prices.loc[:, ~stock_prices.columns.duplicated()]
    stock_prices = stock_prices.sort_index()
    stock_prices = normalize_ticker_columns(stock_prices, valid_tickers)

    stock_returns = stock_prices.pct_change(fill_method=None)
    stock_returns = normalize_ticker_columns(stock_returns, valid_tickers)

    data_source_note = "Downloaded from yfinance using yf.download(auto_adjust=True)"

    save_df_with_date_index(
        stock_prices,
        METADATA_DIR / "downloaded_stock_prices_yfinance_auto_adjust_true.csv"
    )
    save_df_with_date_index(
        stock_returns,
        METADATA_DIR / "downloaded_stock_returns_yfinance_auto_adjust_true.csv"
    )
    pd.DataFrame(failed_batches, columns=["batch_start", "tickers", "error"]).to_csv(
        METADATA_DIR / "failed_yfinance_batches.csv",
        index=False
    )

# Save/copy returns matrix into required metadata output
save_df_with_date_index(
    stock_returns,
    METADATA_DIR / "downloaded_stock_returns_yfinance_auto_adjust_true.csv"
)

# Save/copy prices matrix into required metadata output if available
if prices_source_path is not None:
    try:
        stock_prices_existing = read_csv_with_date(prices_source_path)
        stock_prices_existing = normalize_ticker_columns(stock_prices_existing, valid_tickers)
        save_df_with_date_index(
            stock_prices_existing,
            METADATA_DIR / "downloaded_stock_prices_yfinance_auto_adjust_true.csv"
        )
    except Exception as e:
        pd.DataFrame({
            "note": [
                f"Price matrix source existed but could not be read. "
                f"Experiment used returns matrix. Error: {e}"
            ]
        }).to_csv(
            METADATA_DIR / "downloaded_stock_prices_yfinance_auto_adjust_true.csv",
            index=False
        )
else:
    price_output_path = METADATA_DIR / "downloaded_stock_prices_yfinance_auto_adjust_true.csv"
    if not price_output_path.exists():
        pd.DataFrame({
            "note": [
                "No price matrix was available from inputs. "
                "Experiment used existing returns matrix."
            ]
        }).to_csv(price_output_path, index=False)

# Robust failed_yfinance_batches.csv handling
failed_output_path = METADATA_DIR / "failed_yfinance_batches.csv"

if failed_batches_source_path is not None:
    failed_df = safe_read_csv(
        failed_batches_source_path,
        default_columns=["batch_start", "tickers", "error"]
    )

    if "read_error" in failed_df.columns:
        failed_df = pd.DataFrame([{
            "batch_start": "",
            "tickers": "",
            "error": str(failed_df.iloc[0].get("read_error", "Unknown read error"))
        }])

    for col in ["batch_start", "tickers", "error"]:
        if col not in failed_df.columns:
            failed_df[col] = ""

    failed_df = failed_df[["batch_start", "tickers", "error"]]
    failed_df.to_csv(failed_output_path, index=False)

else:
    if not failed_output_path.exists():
        pd.DataFrame(columns=["batch_start", "tickers", "error"]).to_csv(
            failed_output_path,
            index=False
        )

# Date filtering
stock_returns.index = pd.to_datetime(stock_returns.index, errors="coerce")
stock_returns = stock_returns[stock_returns.index.notna()]
stock_returns = stock_returns.sort_index()
stock_returns = stock_returns[~stock_returns.index.duplicated(keep="first")]

if start_date is not None:
    stock_returns = stock_returns.loc[stock_returns.index >= pd.to_datetime(start_date)]

if end_date is not None:
    stock_returns = stock_returns.loc[stock_returns.index <= pd.to_datetime(end_date)]

stock_returns = stock_returns.loc[:, stock_returns.notna().sum(axis=0) > WINDOW_LOOKBACK]

if stock_returns.shape[1] == 0:
    raise ValueError(
        "No stock return columns have enough non-missing observations after filtering. "
        "Need stock-level daily returns with more than 504 valid observations."
    )

if stock_returns.shape[0] <= WINDOW_LOOKBACK + H:
    raise ValueError(
        f"Not enough trading days after filtering. "
        f"Rows={stock_returns.shape[0]}, required more than {WINDOW_LOOKBACK + H}."
    )

print("\nReturn matrix ready:")
print(f"- shape: {stock_returns.shape}")
print(f"- first date: {stock_returns.index.min().date()}")
print(f"- last date: {stock_returns.index.max().date()}")
print(f"- data source note: {data_source_note}")

# ============================================================
# 5. LOAD BEST PCA-KMEANS AND BEST MAPPER CONFIGURATIONS
# ============================================================

ranking_path = find_file(EXP1_DIR, "step4_full_balanced_score_ranking.csv")

if ranking_path is None:
    ranking_path = find_file(EXP_INPUT_DIR, "step3_all_configurations_scored.csv")

if ranking_path is None:
    raise FileNotFoundError(
        "Could not find step4_full_balanced_score_ranking.csv in Exp1 outputs "
        "or fallback step3_all_configurations_scored.csv in experiment_input."
    )

ranking_df = pd.read_csv(ranking_path)

required_rep_cols = [
    "Method",
    "Annualized Return",
    "Daily Sharpe",
    "Daily Max Drawdown",
    "Average Turnover"
]

missing_rep_cols = [c for c in required_rep_cols if c not in ranking_df.columns]
if missing_rep_cols:
    raise ValueError(
        f"Representative ranking file is missing required columns: {missing_rep_cols}. "
        f"Ranking path: {ranking_path}"
    )

if "Balanced Score Rank" in ranking_df.columns:
    ranking_df = ranking_df.sort_values("Balanced Score Rank", ascending=True)
elif "Balanced Score" in ranking_df.columns:
    ranking_df = ranking_df.sort_values("Balanced Score", ascending=False)

core_df = ranking_df.copy()

if "Rank Population" in core_df.columns:
    core_candidate = core_df[
        core_df["Rank Population"].astype(str).str.lower().str.contains("core", na=False)
    ]
    if not core_candidate.empty:
        core_df = core_candidate

method_lower = core_df["Method"].astype(str).str.lower()

best_pca_df = core_df[
    method_lower.str.replace("-", "", regex=False).str.replace("_", "", regex=False).str.contains("pcakmeans", na=False)
]

if best_pca_df.empty:
    best_pca_df = core_df[method_lower.str.contains("pca", na=False) & method_lower.str.contains("kmeans", na=False)]

best_mapper_df = core_df[method_lower.str.contains("mapper", na=False)]

if best_pca_df.empty:
    raise ValueError("Could not identify best PCA-KMeans configuration from ranking source.")

if best_mapper_df.empty:
    raise ValueError("Could not identify best Mapper configuration from ranking source.")

best_pca = best_pca_df.head(1).iloc[0].to_dict()
best_mapper = best_mapper_df.head(1).iloc[0].to_dict()

print("\nBest PCA-KMeans source row:")
cols_preview = [
    c for c in [
        "Strategy",
        "Method",
        "Rebalance Frequency",
        "l",
        "Annualized Return",
        "Daily Sharpe",
        "Daily Max Drawdown",
        "Average Turnover"
    ]
    if c in ranking_df.columns
]
print(pd.DataFrame([best_pca])[cols_preview].to_string(index=False))

print("\nBest Mapper source row:")
print(pd.DataFrame([best_mapper])[cols_preview].to_string(index=False))

# ============================================================
# 6. RANDOM REPRESENTATIVE-DATE BACKTEST
# ============================================================

def select_random_date_portfolio(train_rets: pd.DataFrame, eligible_stocks, rng: np.random.Generator):
    eligible_stocks = list(eligible_stocks)

    if len(eligible_stocks) == 0:
        return [], []

    if len(train_rets) < K:
        return [], []

    random_positions = rng.choice(np.arange(len(train_rets)), size=K, replace=False)
    selected_union = set()
    selected_dates = []

    for pos in random_positions:
        random_day = train_rets.index[int(pos)]
        selected_dates.append(random_day)

        same_day_returns = train_rets.loc[random_day, eligible_stocks].dropna()

        if len(same_day_returns) == 0:
            continue

        n_select = int(math.ceil(L * len(same_day_returns)))
        n_select = max(n_select, 1)

        top_stocks = same_day_returns.sort_values(ascending=False).head(n_select).index.tolist()
        selected_union.update(top_stocks)

    return sorted(selected_union), selected_dates

def run_single_random_trial(seed: int):
    rng = np.random.default_rng(seed)

    returns = stock_returns.copy()
    dates = returns.index
    n_dates = len(dates)

    rebalance_indices = list(range(WINDOW_LOOKBACK, n_dates, H))
    rebalance_index_set = set(rebalance_indices)

    old_weights = {}
    current_weights = {}

    fallback_count = 0
    nav_previous_day_close = initial_nav

    daily_rows = []
    nav_rows = []
    rebalance_rows = []

    rebalance_turnovers = []
    selected_counts = []

    for idx in range(WINDOW_LOOKBACK, n_dates):
        current_date = dates[idx]

        nav_before_day = nav_previous_day_close
        nav_after_rebalance_cost = nav_before_day

        is_rebalance = idx in rebalance_index_set

        transaction_cost = 0.0
        turnover = 0.0

        eligible_count = np.nan
        selected_random_dates_str = ""
        fallback_used = False
        fallback_reason = ""
        selected_stocks = list(current_weights.keys())

        if is_rebalance:
            train_rets = returns.iloc[idx - WINDOW_LOOKBACK:idx].copy()

            eligible_stocks = train_rets.columns[
                train_rets.notna().sum(axis=0) == WINDOW_LOOKBACK
            ].tolist()

            eligible_count = len(eligible_stocks)

            selected_stocks, random_days = select_random_date_portfolio(
                train_rets=train_rets,
                eligible_stocks=eligible_stocks,
                rng=rng
            )

            selected_random_dates_str = "|".join([
                pd.Timestamp(d).strftime("%Y-%m-%d") for d in random_days
            ])

            if len(selected_stocks) == 0:
                fallback_count += 1
                fallback_used = True

                if len(current_weights) > 0:
                    selected_stocks = list(current_weights.keys())
                    fallback_reason = "empty_random_union_carry_forward_previous_portfolio"
                else:
                    selected_stocks = []
                    fallback_reason = "empty_random_union_no_previous_portfolio_hold_cash"

            new_weights = equal_weights(selected_stocks)

            turnover = one_way_turnover(new_weights, old_weights)

            transaction_cost = nav_before_day * turnover * TRANSACTION_COST_RATE
            nav_after_rebalance_cost = nav_before_day - transaction_cost

            old_weights = new_weights.copy()
            current_weights = new_weights.copy()

            rebalance_turnovers.append(turnover)
            selected_counts.append(len(selected_stocks))

            rebalance_rows.append({
                "Date": current_date,
                "seed": seed,
                "strategy": "Random Representative-Date Benchmark",
                "method": "Random Representative-Date",
                "h": H,
                "l": L,
                "k": K,
                "window_lookback": WINDOW_LOOKBACK,
                "eligible_ticker_count": eligible_count,
                "selected_stock_count": len(selected_stocks),
                "turnover": turnover,
                "transaction_cost": transaction_cost,
                "nav_before_rebalance_cost": nav_before_day,
                "nav_after_rebalance_cost": nav_after_rebalance_cost,
                "random_selected_dates": selected_random_dates_str,
                "fallback_used": fallback_used,
                "fallback_reason": fallback_reason,
                "selected_stocks": "|".join(selected_stocks)
            })

        if len(current_weights) == 0:
            holding_return_before_cost = 0.0
        else:
            held_stocks = list(current_weights.keys())
            day_rets = returns.iloc[idx][held_stocks]
            valid_day_rets = day_rets.dropna()

            if len(valid_day_rets) == 0:
                holding_return_before_cost = 0.0
            else:
                # Equal-weight over available held stocks for that trading day.
                # No forward-fill/backward-fill is used.
                holding_return_before_cost = float(valid_day_rets.mean())

        nav_after_day = nav_after_rebalance_cost * (1.0 + holding_return_before_cost)

        daily_return_after_cost = nav_after_day / nav_previous_day_close - 1.0

        daily_rows.append({
            "Date": current_date,
            "seed": seed,
            "strategy": "Random Representative-Date Benchmark",
            "daily_return_before_cost": holding_return_before_cost,
            "daily_return_after_cost": daily_return_after_cost,
            "transaction_cost_on_date": transaction_cost,
            "turnover_on_date": turnover,
            "is_rebalance_date": bool(is_rebalance),
            "NAV_before_day": nav_before_day,
            "NAV_after_rebalance_cost": nav_after_rebalance_cost,
            "NAV_after_day": nav_after_day
        })

        nav_rows.append({
            "Date": current_date,
            "seed": seed,
            "strategy": "Random Representative-Date Benchmark",
            "NAV": nav_after_day
        })

        nav_previous_day_close = nav_after_day

    daily_df = pd.DataFrame(daily_rows)
    nav_df = pd.DataFrame(nav_rows)
    rebalance_df = pd.DataFrame(rebalance_rows)

    after_cost_returns = daily_df["daily_return_after_cost"].astype(float)
    nav_series = daily_df["NAV_after_day"].astype(float)

    cumulative_return = float(nav_series.iloc[-1] / initial_nav - 1.0)
    cumulative_from_returns = float((1.0 + after_cost_returns).prod() - 1.0)
    consistency_error = abs(cumulative_return - cumulative_from_returns)

    metrics = {
        "seed": seed,
        "strategy": "Random Representative-Date Benchmark",
        "method": "Random Representative-Date",
        "h": H,
        "l": L,
        "k": K,
        "Cumulative Return": cumulative_return,
        "Annualized Return": annualized_return_from_daily_returns(after_cost_returns),
        "Daily Sharpe": annualized_sharpe_from_daily_returns(after_cost_returns),
        "Daily Max Drawdown": max_drawdown_from_nav(nav_series),
        "Average Turnover": float(np.mean(rebalance_turnovers)) if len(rebalance_turnovers) else np.nan,
        "Average Number of Stocks": float(np.mean(selected_counts)) if len(selected_counts) else np.nan,
        "Rebalance Count": int(len(rebalance_turnovers)),
        "Fallback Count": int(fallback_count),
        "final_nav": float(nav_series.iloc[-1]),
        "cumulative_return_from_daily_returns": cumulative_from_returns,
        "cumulative_return_consistency_error": consistency_error
    }

    return metrics, daily_df, nav_df, rebalance_df

all_metrics = []
all_daily_returns = []
all_nav = []
all_rebalance_logs = []

for trial_idx in range(N_RANDOM_TRIALS):
    seed = SEED_BASE + trial_idx

    metrics, daily_df, nav_df, rebalance_df = run_single_random_trial(seed)

    if metrics["cumulative_return_consistency_error"] > CONSISTENCY_TOL:
        raise RuntimeError(
            f"Metric consistency error exceeds tolerance for seed={seed}: "
            f"{metrics['cumulative_return_consistency_error']}"
        )

    all_metrics.append(metrics)
    all_daily_returns.append(daily_df)
    all_nav.append(nav_df)
    all_rebalance_logs.append(rebalance_df)

metrics_df = pd.DataFrame(all_metrics)
daily_returns_df = pd.concat(all_daily_returns, ignore_index=True)
daily_nav_df = pd.concat(all_nav, ignore_index=True)
rebalance_log_df = pd.concat(all_rebalance_logs, ignore_index=True)

# ============================================================
# 7. SUMMARY AND PERCENTILES
# ============================================================

summary_metrics = [
    "Cumulative Return",
    "Annualized Return",
    "Daily Sharpe",
    "Daily Max Drawdown",
    "Average Turnover",
    "Average Number of Stocks"
]

summary_rows = []

for metric_name in summary_metrics:
    s = metrics_df[metric_name].replace([np.inf, -np.inf], np.nan).dropna().astype(float)

    summary_rows.append({
        "metric": metric_name,
        "mean": s.mean(),
        "median": s.median(),
        "std": s.std(ddof=1),
        "p05": s.quantile(0.05),
        "p25": s.quantile(0.25),
        "p75": s.quantile(0.75),
        "p95": s.quantile(0.95),
        "min": s.min(),
        "max": s.max()
    })

summary_df = pd.DataFrame(summary_rows)

percentile_rows = []

for metric_name in summary_metrics:
    s = metrics_df[metric_name].replace([np.inf, -np.inf], np.nan).dropna().astype(float)

    percentile_rows.append({
        "metric": metric_name,
        "p05": s.quantile(0.05),
        "p25": s.quantile(0.25),
        "p50": s.quantile(0.50),
        "p75": s.quantile(0.75),
        "p95": s.quantile(0.95)
    })

percentiles_df = pd.DataFrame(percentile_rows)

# ============================================================
# 8. COMPARISON VS PCA-KMEANS AND MAPPER
# ============================================================

comparison_metrics = [
    "Annualized Return",
    "Daily Sharpe",
    "Daily Max Drawdown",
    "Average Turnover"
]

higher_is_better_map = {
    "Annualized Return": True,
    "Daily Sharpe": True,
    "Daily Max Drawdown": True,
    "Average Turnover": False
}

def comparison_table(rep_row: dict, target_name: str):
    rows = []

    for metric_name in comparison_metrics:
        random_s = metrics_df[metric_name].replace([np.inf, -np.inf], np.nan).dropna().astype(float)

        rep_value = float(rep_row[metric_name])
        higher_is_better = higher_is_better_map[metric_name]

        percentile = percentile_vs_distribution(
            value=rep_value,
            distribution=random_s,
            higher_is_better=higher_is_better
        )

        if metric_name == "Average Turnover":
            note = (
                f"Lower is better. Representative Average Turnover is at the {percentile:.1f} percentile "
                "versus corrected random-date trials, where a higher percentile means lower/equal turnover than more random trials."
            )
        else:
            note = (
                f"Higher is better. Representative {metric_name} is at the {percentile:.1f} percentile "
                "versus corrected random-date trials."
            )

        rows.append({
            "comparison_target": metric_name,
            "representative_strategy": rep_row.get("Strategy", target_name),
            "representative_method": rep_row.get("Method", target_name),
            "representative_h": rep_row.get("Rebalance Frequency", np.nan),
            "representative_l": rep_row.get("l", np.nan),
            "representative_annualized_return": rep_row.get("Annualized Return", np.nan),
            "representative_daily_sharpe": rep_row.get("Daily Sharpe", np.nan),
            "representative_daily_max_drawdown": rep_row.get("Daily Max Drawdown", np.nan),
            "representative_average_turnover": rep_row.get("Average Turnover", np.nan),
            "random_metric_mean": random_s.mean(),
            "random_metric_median": random_s.median(),
            "random_metric_p05": random_s.quantile(0.05),
            "random_metric_p25": random_s.quantile(0.25),
            "random_metric_p75": random_s.quantile(0.75),
            "random_metric_p95": random_s.quantile(0.95),
            "representative_minus_random_mean": rep_value - random_s.mean(),
            "representative_percentile_vs_random_distribution": percentile,
            "interpretation_note": note
        })

    return pd.DataFrame(rows)

vs_pca_df = comparison_table(best_pca, "PCA-KMeans")
vs_mapper_df = comparison_table(best_mapper, "Mapper")

# ============================================================
# 9. SAVE MAIN OUTPUT FILES
# ============================================================

metrics_df.to_csv(OUTPUT_DIR / "step4_random_date_trials_metrics.csv", index=False)
summary_df.to_csv(OUTPUT_DIR / "step4_random_date_summary.csv", index=False)
percentiles_df.to_csv(OUTPUT_DIR / "step4_random_date_percentiles.csv", index=False)
vs_pca_df.to_csv(OUTPUT_DIR / "step4_random_date_vs_pca_kmeans.csv", index=False)
vs_mapper_df.to_csv(OUTPUT_DIR / "step4_random_date_vs_mapper.csv", index=False)
rebalance_log_df.to_csv(OUTPUT_DIR / "step4_random_date_rebalance_log.csv", index=False)
daily_nav_df.to_csv(OUTPUT_DIR / "step4_random_date_daily_nav.csv", index=False)
daily_returns_df.to_csv(OUTPUT_DIR / "step4_random_date_daily_returns.csv", index=False)

# ============================================================
# 10. SAVE RUN CONFIG
# ============================================================

max_consistency_error = float(metrics_df["cumulative_return_consistency_error"].abs().max())

run_config = {
    "step": "Step 4",
    "experiment": "Experiment 3 - Corrected Random Representative-Date Benchmark",
    "generated_at": datetime.now().isoformat(timespec="seconds"),
    "input_files": {
        "experiment_input.zip_present": EXPERIMENT_INPUT_ZIP.exists(),
        "experiment_input.zip_sha256": sha256_file(EXPERIMENT_INPUT_ZIP),
        "tda_step4_exp1_outputs.zip_present": EXP1_ZIP.exists(),
        "tda_step4_exp1_outputs.zip_sha256": sha256_file(EXP1_ZIP),
        "tda_step4_exp2_outputs.zip_present": EXP2_ZIP.exists(),
        "tda_step4_exp2_outputs.zip_sha256": sha256_file(EXP2_ZIP)
    },
    "core_configuration": {
        "h": H,
        "l": L,
        "k": K,
        "window_lookback": WINDOW_LOOKBACK,
        "transaction_cost_rate": TRANSACTION_COST_RATE,
        "n_random_trials": N_RANDOM_TRIALS,
        "seed_base": SEED_BASE,
        "initial_nav": initial_nav
    },
    "data": {
        "start_date": start_date,
        "end_date": end_date,
        "valid_ticker_count_from_step1_metadata": len(valid_tickers),
        "return_matrix_shape_used": list(stock_returns.shape),
        "returns_source_path": str(returns_source_path) if returns_source_path is not None else "",
        "prices_source_path": str(prices_source_path) if prices_source_path is not None else "",
        "data_source_note": data_source_note,
        "missing_data_rule": (
            "At each rebalance, ticker eligible only if it has full non-missing returns "
            "in 504-day lookback. No forward-fill/backward-fill of returns."
        )
    },
    "metric_consistency": {
        "metrics_use": "daily_return_after_cost",
        "max_abs_cumulative_return_consistency_error": max_consistency_error,
        "tolerance": CONSISTENCY_TOL,
        "passed": bool(max_consistency_error <= CONSISTENCY_TOL)
    },
    "representative_comparison_source": {
        "ranking_path": str(ranking_path),
        "best_pca_kmeans_strategy": best_pca.get("Strategy", ""),
        "best_mapper_strategy": best_mapper.get("Strategy", "")
    },
    "guardrails": [
        "Random-date benchmark tests whether representative-date extraction adds value beyond arbitrary historical day selection.",
        "If PCA-KMeans/Mapper outperform random-date distribution, state this as empirical evidence within the tested sample.",
        "Do not claim statistical superiority until bootstrap experiment is completed.",
        "Do not claim investability or proven alpha."
    ]
}

with open(METADATA_DIR / "run_config.json", "w", encoding="utf-8") as f:
    json.dump(run_config, f, indent=2)

# ============================================================
# 11. SAVE AUDIT MARKDOWN
# ============================================================

fallback_summary_text = metrics_df["Fallback Count"].describe().to_string()

audit_md = f"""# Step 4 Experiment 3 Audit — Corrected Random Representative-Date Benchmark

## Purpose

This experiment tests whether representative-date extraction in PCA-KMeans/Mapper adds value beyond arbitrary historical day selection.

## Core Configuration

- h = {H}
- l = {L}
- k = {K}
- window lookback = {WINDOW_LOOKBACK}
- transaction cost rate = {TRANSACTION_COST_RATE}
- random trials = {N_RANDOM_TRIALS}
- seed base = {SEED_BASE}
- initial NAV = {initial_nav:,.2f}

## Data Source

- Returns source: `{returns_source_path}`
- Prices source: `{prices_source_path}`
- Return matrix shape used: {stock_returns.shape}
- Step 1 metadata path: `{step1_meta_path}`
- Data source note: {data_source_note}

## Corrected Metric Logic

Transaction cost is subtracted from NAV at each rebalance date before applying the day's holding return.

The daily return used for all performance metrics is:

`daily_return_after_cost = NAV_after_day / NAV_previous_day_close - 1`

The following metrics are all calculated from the same after-cost NAV path / after-cost daily returns:

- Cumulative Return
- Annualized Return
- Daily Sharpe
- Daily Max Drawdown

## Metric Consistency Check

- max_abs_cumulative_return_consistency_error = {max_consistency_error:.12g}
- tolerance = {CONSISTENCY_TOL}
- status = {"PASS" if max_consistency_error <= CONSISTENCY_TOL else "FAIL"}

## Random Trial Summary

{safe_to_markdown(summary_df, index=False)}

## Fallback Count Summary

{fallback_summary_text}

## Best PCA-KMeans Compared

Representative strategy: {best_pca.get("Strategy", "")}

{safe_to_markdown(vs_pca_df, index=False)}

## Best Mapper Compared

Representative strategy: {best_mapper.get("Strategy", "")}

{safe_to_markdown(vs_mapper_df, index=False)}

## Interpretation Guardrails

- Random-date benchmark tests whether representative-date extraction adds value beyond arbitrary historical day selection.
- If PCA-KMeans/Mapper outperform random-date distribution, state this as empirical evidence within the tested sample.
- Do not claim statistical superiority until bootstrap experiment is completed.
- Do not claim investability or proven alpha.
"""

with open(OUTPUT_DIR / "step4_random_date_audit.md", "w", encoding="utf-8") as f:
    f.write(audit_md)

# ============================================================
# 12. ARTIFACT CHECK AND ZIP
# ============================================================

required_files = [
    "step4_random_date_trials_metrics.csv",
    "step4_random_date_summary.csv",
    "step4_random_date_percentiles.csv",
    "step4_random_date_vs_pca_kmeans.csv",
    "step4_random_date_vs_mapper.csv",
    "step4_random_date_audit.md",
    "step4_random_date_rebalance_log.csv",
    "step4_random_date_daily_nav.csv",
    "step4_random_date_daily_returns.csv",
    "step4_exp3_artifact_file_check.csv",
    "00_metadata/run_config.json",
    "00_metadata/downloaded_stock_prices_yfinance_auto_adjust_true.csv",
    "00_metadata/downloaded_stock_returns_yfinance_auto_adjust_true.csv",
    "00_metadata/failed_yfinance_batches.csv"
]

artifact_check_df = make_artifact_check(OUTPUT_DIR, required_files)
artifact_check_df.to_csv(OUTPUT_DIR / "step4_exp3_artifact_file_check.csv", index=False)

artifact_check_df = make_artifact_check(OUTPUT_DIR, required_files)
artifact_check_df.to_csv(OUTPUT_DIR / "step4_exp3_artifact_file_check.csv", index=False)

missing_artifacts = artifact_check_df[~artifact_check_df["exists"]]

if not missing_artifacts.empty:
    raise RuntimeError(
        "Missing required artifacts:\n"
        + missing_artifacts.to_string(index=False)
    )

zip_path = Path("tda_step4_exp3_outputs.zip")

if zip_path.exists():
    zip_path.unlink()

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for p in OUTPUT_DIR.rglob("*"):
        if p.is_file():
            z.write(p, arcname=str(p.relative_to(OUTPUT_DIR.parent)))

# ============================================================
# 13. REQUIRED NOTEBOOK PREVIEWS
# ============================================================

print("\n" + "=" * 100)
print("ARTIFACT CHECK TABLE")
print("=" * 100)
print(artifact_check_df.to_string(index=False))

print("\n" + "=" * 100)
print("SUMMARY STATISTICS PREVIEW")
print("=" * 100)
print(summary_df.to_string(index=False))

print("\n" + "=" * 100)
print("BEST RANDOM TRIALS PREVIEW BY DAILY SHARPE")
print("=" * 100)
print(
    metrics_df.sort_values("Daily Sharpe", ascending=False).head(10)[[
        "seed",
        "Cumulative Return",
        "Annualized Return",
        "Daily Sharpe",
        "Daily Max Drawdown",
        "Average Turnover",
        "Average Number of Stocks",
        "Rebalance Count",
        "Fallback Count"
    ]].to_string(index=False)
)

print("\n" + "=" * 100)
print("WORST RANDOM TRIALS PREVIEW BY DAILY SHARPE")
print("=" * 100)
print(
    metrics_df.sort_values("Daily Sharpe", ascending=True).head(10)[[
        "seed",
        "Cumulative Return",
        "Annualized Return",
        "Daily Sharpe",
        "Daily Max Drawdown",
        "Average Turnover",
        "Average Number of Stocks",
        "Rebalance Count",
        "Fallback Count"
    ]].to_string(index=False)
)

print("\n" + "=" * 100)
print("COMPARISON VS PCA-KMEANS PREVIEW")
print("=" * 100)
print(vs_pca_df.to_string(index=False))

print("\n" + "=" * 100)
print("COMPARISON VS MAPPER PREVIEW")
print("=" * 100)
print(vs_mapper_df.to_string(index=False))

print("\n" + "=" * 100)
print("FALLBACK COUNT SUMMARY")
print("=" * 100)
print(metrics_df["Fallback Count"].describe().to_string())

print("\n" + "=" * 100)
print("METRIC CONSISTENCY CHECK")
print("=" * 100)
print(f"max_abs_cumulative_return_consistency_error = {max_consistency_error:.12g}")
print(f"tolerance = {CONSISTENCY_TOL}")
print(f"status = {'PASS' if max_consistency_error <= CONSISTENCY_TOL else 'FAIL'}")

print("\n" + "=" * 100)
print("OUTPUT ZIP CREATED")
print("=" * 100)
print(str(zip_path.resolve()))
print(f"ZIP SHA256: {sha256_file(zip_path)}")

Input extraction completed.
- experiment_input extracted to: _step4_exp3_work\experiment_input_extracted
- exp1 outputs extracted to: _step4_exp3_work\exp1_extracted
- exp2 outputs extracted to: _step4_exp3_work\exp2_extracted

Loaded Step 1 metadata:
- step1 metadata path: _step4_exp3_work\experiment_input_extracted\experiment_input\01_step1_mapper\tda_experiment_outputs_step1_extracted\tda_experiment_outputs_step1\step1_experiment_metadata.json
- valid ticker count: 359
- start_date: 2014-01-01
- end_date: 2025-12-31
- initial_nav: 1000000.0

Loaded stock returns from: _step4_exp3_work\exp2_extracted\00_metadata\downloaded_stock_returns_yfinance_auto_adjust_true.csv

Return matrix ready:
- shape: (3016, 359)
- first date: 2014-01-03
- last date: 2025-12-30
- data source note: Loaded existing returns matrix from _step4_exp3_work\exp2_extracted\00_metadata\downloaded_stock_returns_yfinance_auto_adjust_true.csv

Best PCA-KMeans source row:
               Strategy     Method  Rebalance F